# Composer Classification of MIDI Music with LSTM and CNN Models

**Deep Learning Final Project**

This notebook develops and compares two deep learning models - a **Long Short-Term Memory (LSTM)** network and a **Convolutional Neural Network (CNN)** - that predict the composer of a MIDI musical composition. The task is restricted to four composers:

1. **Bach**
2. **Beethoven**
3. **Chopin**
4. **Mozart**

## Project overview

Each architecture receives a representation that suits it. The LSTM consumes *ordered note-pitch sequences* (an event stream), which matches its strength at modeling temporal order. The CNN consumes *piano-roll matrices* (a 2-D pitch-by-time image), which matches its strength at detecting local 2-D patterns such as chords and textures. Both models are trained, tuned, and evaluated on the same train/validation/test partition, and their results are compared directly.

## How to run this notebook

**Google Colab (recommended):**
1. `Runtime -> Change runtime type -> GPU` (optional but faster).
2. Run the *Setup* cells to install libraries.
3. Provide the dataset (see the **Dataset loading** section). Either upload a Kaggle API token to auto-download, mount Google Drive, or upload the archive manually.
4. Set `DATASET_PATH` to the folder that contains the composer sub-folders.
5. Run the notebook top to bottom (`Runtime -> Run all`).

**Local Jupyter:** the same steps apply, minus Google Drive mounting. Everything is CPU-compatible; a GPU only speeds up training.

> **Reproducibility & honesty note.** All metrics, tables, and figures in this notebook are produced by the code at run time. Nothing is hard-coded. Cells that print final numbers are marked so you can copy the printed values into the report. If a step cannot run (for example, the dataset is missing), the code fails loudly with a clear message rather than silently continuing.


## 1. Setup: install and import libraries

The next cell installs the third-party libraries the notebook needs. On Colab most are pre-installed; `pretty_midi` and `music21` usually are not. The install cell is safe to re-run.

In [ ]:
# --- Install libraries (safe to re-run). Comment out if already installed. ---
import sys, subprocess

def _pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# pretty_midi is the primary MIDI parser; music21 is imported only as a fallback
# demonstration. kaggle is optional (used for automatic download).
_pip_install(["pretty_midi==0.2.10", "music21", "scikit-learn",
              "seaborn", "kaggle"])
print("Setup cell finished. If imports below fail, restart the runtime and re-run.")

In [ ]:
# --- Imports ---
import os, re, glob, hashlib, random, warnings, collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MIDI parsing (with graceful failure if unavailable)
try:
    import pretty_midi
except ImportError as e:
    raise ImportError(
        "pretty_midi is required. Run the install cell above, then restart the "
        "runtime and re-run."
    ) from e

# Deep learning
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
pretty_midi.pretty_midi.MAX_TICK = 1e7  # tolerate long files without warnings
sns.set_context("notebook")
print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 2. Configuration and random seeds

All tunable constants live here so the notebook has a single control panel. `DATASET_PATH` is the **only** path you normally need to change. Random seeds are fixed for NumPy, Python, and TensorFlow so runs are reproducible (small non-determinism can still occur on GPU).

In [ ]:
# ----------------------------- CONFIG --------------------------------------
DATASET_PATH = r"/Users/robertshifrin/Documents/Masters Classes Folder/Neural Networks and Deep Learning/archive/dataset_4composers"   # local Mac default (clean 4-composer folder).
# In Google Colab, change this after loading the data, e.g. "/content/dataset".

TARGET_COMPOSERS = ["Bach", "Beethoven", "Chopin", "Mozart"]

# LSTM (note-sequence) settings
SEQ_LEN      = 100        # notes per training window
SEQ_STRIDE   = 50         # hop between windows (controls overlap / augmentation)
PITCH_VOCAB  = 128        # MIDI pitches 0-127
PAD_TOKEN    = 128        # reserved padding index -> embedding input_dim = 129

# CNN (piano-roll) settings
PR_FS        = 8          # piano-roll sampling frequency (frames per second)
PR_TIME      = 128        # time frames per window
PR_PITCH     = 128        # pitch rows (fixed by MIDI)
PR_STRIDE    = 64         # hop between piano-roll windows

# Split / training settings
TEST_SIZE    = 0.15
VAL_SIZE     = 0.15       # fraction of the WHOLE dataset (taken from train remainder)
BATCH_SIZE   = 64
MAX_EPOCHS   = 60
PATIENCE     = 8
SEED         = 42

# Data-augmentation (training split only)
AUG_TRANSPOSITIONS = [-2, -1, 1, 2]   # semitone shifts; [] disables transposition

# Optional caps to keep runtime reasonable on limited hardware.
# Set to None to use every file / window.
MAX_FILES_PER_COMPOSER   = None
MAX_WINDOWS_PER_FILE     = 20
# ---------------------------------------------------------------------------

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

set_seeds()
LABEL2IDX = {c: i for i, c in enumerate(TARGET_COMPOSERS)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}
print("Label encoding:", LABEL2IDX)

## 3. Dataset loading

**Dataset:** *midi_classic_music* by Bohdan Fedorak, hosted on Kaggle - 3,929 MIDI files from 175 composers (Fedorak, 2019). The uploaded `midi-classic-music-metadata.json` is the Croissant descriptor for this dataset; it supplies the source URL and citation but does **not** contain the audio files, so the files must be downloaded.

Choose **one** of the three loading options below. After loading, `DATASET_PATH` must point to a directory that contains the composer sub-folders (or any nested structure that includes the composer names in the path - the parser searches recursively).

In [ ]:
# --- Option A: automatic Kaggle download (needs a kaggle.json API token) ------
# In Colab: upload kaggle.json first, e.g. via files.upload(), then run this cell.
#
# from google.colab import files; files.upload()   # choose kaggle.json
# os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
# import shutil; shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
# os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
#
# !kaggle datasets download -d blanderbuss/midi-classic-music -p /content --unzip
# DATASET_PATH = "/content/midiclassics"   # adjust to the unzipped folder name
print("Option A is commented out. Uncomment to auto-download via the Kaggle API.")

In [ ]:
# --- Option B: mount Google Drive (if you saved the dataset there) ------------
# from google.colab import drive; drive.mount('/content/drive')
# DATASET_PATH = "/content/drive/MyDrive/midi_classic_music"
print("Option B is commented out. Uncomment to mount Google Drive.")

In [ ]:
# --- Option C: manual upload of a zip, then unzip -----------------------------
# from google.colab import files
# up = files.upload()                       # choose your archive.zip
# import zipfile
# with zipfile.ZipFile(next(iter(up)), 'r') as z:
#     z.extractall('/content/dataset')
# DATASET_PATH = "/content/dataset"
print("Option C is commented out. Uncomment to upload and unzip manually.")

### 3.1 (Optional) Extract nested `.zip` archives

Some composer folders in the original dataset contain nested `.zip` files that hold extra MIDI files (for example, several Beethoven sonatas). The clean `dataset_4composers` folder already has these extracted, so this cell is a no-op there. Run it only if you point `DATASET_PATH` at a copy that still contains nested zips; it extracts any it finds, in place.

In [ ]:
# Optional: extract any nested .zip archives found under DATASET_PATH (safe to re-run).
import zipfile
from pathlib import Path

def extract_nested_zips(root):
    root = Path(root)
    zips = list(root.rglob('*.zip'))
    if not zips:
        print('No nested .zip archives found. Nothing to extract.')
        return
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent / ('_extracted_' + z.stem))
            print('Extracted:', z)
        except zipfile.BadZipFile:
            print('Skipped (not a valid zip):', z)

extract_nested_zips(DATASET_PATH)

## 4. Dataset inspection, filtering, and label creation

Rather than assume a folder layout, the code **walks `DATASET_PATH` recursively**, collects every `.mid`/`.midi` file, and infers the composer from the file path. A file is kept only if exactly one of the four target composer names appears in its path - this rejects ambiguous files and every non-target composer. This step also performs **de-duplication by MD5 hash** so identical files cannot leak across splits.

In [ ]:
def find_midi_files(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            f"DATASET_PATH does not exist: {root}\n"
            "Load the dataset (Section 3) and set DATASET_PATH correctly.")
    files = []
    for ext in ("*.mid", "*.midi", "*.MID", "*.MIDI"):
        files.extend(root.rglob(ext))
    return sorted(set(files))

def infer_composer(path, targets=TARGET_COMPOSERS):
    """Return the single matching target composer, or None if 0 or >1 match."""
    text = str(path).lower()
    hits = [c for c in targets
            if re.search(r"(^|[^a-z])" + c.lower() + r"([^a-z]|$)", text)]
    return hits[0] if len(hits) == 1 else None

all_files = find_midi_files(DATASET_PATH)
print(f"Total MIDI files found under DATASET_PATH: {len(all_files)}")
if len(all_files) == 0:
    raise RuntimeError("No MIDI files found. Check DATASET_PATH and the dataset contents.")

records = []
for f in all_files:
    comp = infer_composer(f)
    if comp is not None:
        records.append({"path": str(f), "composer": comp})

raw_df = pd.DataFrame(records)
print(f"Files matching the four target composers: {len(raw_df)}")
print("\nRaw counts per composer (before cleaning):")
print(raw_df["composer"].value_counts())

In [ ]:
# --- De-duplicate by content hash and drop unreadable files ------------------
def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

raw_df["md5"] = raw_df["path"].map(md5_of)
before = len(raw_df)
dedup_df = raw_df.drop_duplicates(subset="md5").reset_index(drop=True)
print(f"Duplicate files removed: {before - len(dedup_df)}")

# Validate that each file is parseable; record note counts.
def probe_midi(path):
    try:
        pm = pretty_midi.PrettyMIDI(path)
        n_notes = sum(len(inst.notes) for inst in pm.instruments)
        return n_notes
    except Exception:
        return -1   # unreadable / invalid

dedup_df["n_notes"] = dedup_df["path"].map(probe_midi)
unreadable = int((dedup_df["n_notes"] < 0).sum())
empty      = int((dedup_df["n_notes"] == 0).sum())
print(f"Unreadable / invalid MIDI files: {unreadable}")
print(f"Readable but empty (no notes):   {empty}")

# Keep only files with a usable number of notes for at least one window.
clean_df = dedup_df[dedup_df["n_notes"] >= SEQ_LEN].reset_index(drop=True)
print(f"\nUsable files (>= {SEQ_LEN} notes): {len(clean_df)}")
print("\nUsable counts per composer (after cleaning):")
print(clean_df["composer"].value_counts())

### 4.1 Before/after summary and class distribution

The table below reports usable file counts per composer **before and after** preprocessing, satisfying the requirement to document data availability explicitly. The bar chart visualizes class balance; any strong imbalance is addressed later with stratified splitting and class weights.

In [ ]:
summary = (pd.DataFrame({
        "raw_matched": raw_df["composer"].value_counts(),
        "after_dedup": dedup_df["composer"].value_counts(),
        "usable":      clean_df["composer"].value_counts(),
    })
    .reindex(TARGET_COMPOSERS)
    .fillna(0).astype(int))
summary.loc["TOTAL"] = summary.sum()
print("File counts per composer:\n")
print(summary)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
clean_df["composer"].value_counts().reindex(TARGET_COMPOSERS).plot(
    kind="bar", ax=ax[0], color=sns.color_palette("deep", 4))
ax[0].set_title("Usable files per composer"); ax[0].set_ylabel("file count")
ax[0].tick_params(axis="x", rotation=0)
clean_df["n_notes"].plot(kind="hist", bins=40, ax=ax[1], color="steelblue")
ax[1].set_title("Distribution of notes per file"); ax[1].set_xlabel("notes")
plt.tight_layout(); plt.show()

## 5. Train / validation / test splitting (leakage-safe)

**Splitting happens at the file level, before any windowing.** If windows from one piece landed in both training and test sets, the model could memorize a piece rather than a composer - a classic data-leakage trap. Splitting whole files first guarantees that every window from a given composition stays inside a single split. The split is **stratified by composer** so each split keeps the same class proportions.

In [ ]:
files = clean_df["path"].values
labels = clean_df["composer"].map(LABEL2IDX).values

# Optional per-composer cap for speed (applied deterministically).
if MAX_FILES_PER_COMPOSER is not None:
    keep_idx = []
    for c in range(len(TARGET_COMPOSERS)):
        idx = np.where(labels == c)[0]
        rng = np.random.RandomState(SEED)
        rng.shuffle(idx)
        keep_idx.extend(idx[:MAX_FILES_PER_COMPOSER])
    keep_idx = np.array(sorted(keep_idx))
    files, labels = files[keep_idx], labels[keep_idx]

# First carve out the test set, then split the remainder into train/val.
f_trainval, f_test, y_trainval, y_test = train_test_split(
    files, labels, test_size=TEST_SIZE, stratify=labels, random_state=SEED)
val_relative = VAL_SIZE / (1.0 - TEST_SIZE)
f_train, f_val, y_train, y_val = train_test_split(
    f_trainval, y_trainval, test_size=val_relative,
    stratify=y_trainval, random_state=SEED)

print(f"Files -> train: {len(f_train)}  val: {len(f_val)}  test: {len(f_test)}")
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    dist = {IDX2LABEL[i]: int((y == i).sum()) for i in range(len(TARGET_COMPOSERS))}
    print(f"  {name:5s} class counts: {dist}")

# Sanity check: no file appears in more than one split.
assert set(f_train) & set(f_test) == set()
assert set(f_train) & set(f_val)  == set()
assert set(f_val)   & set(f_test) == set()
print("Leakage check passed: splits are disjoint at the file level.")

## 6. Feature extraction

Two representations are extracted from the **same** MIDI files, one tailored to each model.

**LSTM - note-pitch sequences.** Every note across all instruments is sorted by onset time, and its MIDI pitch (0-127) becomes a token. The stream is cut into overlapping windows of `SEQ_LEN` notes (hop `SEQ_STRIDE`). Ordered pitch tokens preserve melodic and harmonic motion over time, which is what a recurrent network is built to model. An `Embedding` layer turns each pitch token into a learned dense vector.

**CNN - piano-roll windows.** `pretty_midi.get_piano_roll(fs=PR_FS)` produces a 128 x time matrix of note activity. It is binarized and cut into `PR_PITCH` x `PR_TIME` windows (hop `PR_STRIDE`). A piano roll is effectively an image of the music, so 2-D convolutions can pick up chords (vertical structure) and rhythmic/melodic motifs (horizontal structure).

Because both representations are windowed, one file yields several samples - a form of **sequence-window augmentation** that also enlarges the dataset.

In [ ]:
def extract_note_sequence(path):
    """Return a 1-D array of pitch tokens ordered by onset time, or None."""
    try:
        pm = pretty_midi.PrettyMIDI(path)
    except Exception:
        return None
    notes = [(n.start, n.pitch) for inst in pm.instruments
             if not inst.is_drum for n in inst.notes]
    if len(notes) < SEQ_LEN:
        return None
    notes.sort(key=lambda x: x[0])
    return np.array([p for _, p in notes], dtype=np.int16)

def sequence_windows(seq, seq_len=SEQ_LEN, stride=SEQ_STRIDE, cap=MAX_WINDOWS_PER_FILE):
    out = []
    for start in range(0, len(seq) - seq_len + 1, stride):
        out.append(seq[start:start + seq_len])
        if cap is not None and len(out) >= cap:
            break
    return out

def extract_piano_roll(path):
    """Return a binarized 128 x time piano roll, or None."""
    try:
        pm = pretty_midi.PrettyMIDI(path)
    except Exception:
        return None
    if len(pm.instruments) == 0:
        return None
    roll = pm.get_piano_roll(fs=PR_FS)            # (128, T) velocity
    if roll.shape[1] < PR_TIME:
        return None
    return (roll > 0).astype(np.float32)          # binarize

def roll_windows(roll, t=PR_TIME, stride=PR_STRIDE, cap=MAX_WINDOWS_PER_FILE):
    out = []
    for start in range(0, roll.shape[1] - t + 1, stride):
        out.append(roll[:, start:start + t])
        if cap is not None and len(out) >= cap:
            break
    return out

print("Feature-extraction helpers defined.")

### 6.1 Build windowed datasets per split

The helper below turns a list of files into arrays of windows plus matching labels. **Augmentation (pitch transposition) is applied to the training split only**; validation and test windows are never augmented, preserving an honest estimate of generalization. Transposition by a few semitones changes key but preserves intervals and contour, so the composer's style is retained.

In [ ]:
def transpose_sequence(seq, shift):
    out = seq.astype(np.int16) + shift
    out = np.clip(out, 0, PITCH_VOCAB - 1)
    return out

def transpose_roll(roll, shift):
    out = np.zeros_like(roll)
    if shift == 0:
        return roll
    if shift > 0:
        out[shift:, :] = roll[:-shift, :]
    else:
        out[:shift, :] = roll[-shift:, :]
    return out

def build_sequence_split(files, labels, augment=False):
    X, y = [], []
    skipped = 0
    for path, lab in zip(files, labels):
        seq = extract_note_sequence(path)
        if seq is None:
            skipped += 1
            continue
        for w in sequence_windows(seq):
            X.append(w); y.append(lab)
            if augment:
                for s in AUG_TRANSPOSITIONS:
                    X.append(transpose_sequence(w, s)); y.append(lab)
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32), skipped

def build_roll_split(files, labels, augment=False):
    X, y = [], []
    skipped = 0
    for path, lab in zip(files, labels):
        roll = extract_piano_roll(path)
        if roll is None:
            skipped += 1
            continue
        for w in roll_windows(roll):
            X.append(w); y.append(lab)
            if augment:
                for s in AUG_TRANSPOSITIONS:
                    X.append(transpose_roll(w, s)); y.append(lab)
    X = np.array(X, dtype=np.float32)
    if X.ndim == 3:
        X = X[..., np.newaxis]          # add channel dim -> (N, 128, PR_TIME, 1)
    return X, np.array(y, dtype=np.int32), skipped

print("Split builders defined.")

In [ ]:
# ---- LSTM sequence datasets ----
Xseq_train, yseq_train, sk1 = build_sequence_split(f_train, y_train, augment=True)
Xseq_val,   yseq_val,   sk2 = build_sequence_split(f_val,   y_val,   augment=False)
Xseq_test,  yseq_test,  sk3 = build_sequence_split(f_test,  y_test,  augment=False)
print("LSTM (note-sequence) windows")
print(f"  train: {Xseq_train.shape}  val: {Xseq_val.shape}  test: {Xseq_test.shape}")
print(f"  files skipped (train/val/test): {sk1}/{sk2}/{sk3}")

# ---- CNN piano-roll datasets ----
Xcnn_train, ycnn_train, sk4 = build_roll_split(f_train, y_train, augment=True)
Xcnn_val,   ycnn_val,   sk5 = build_roll_split(f_val,   y_val,   augment=False)
Xcnn_test,  ycnn_test,  sk6 = build_roll_split(f_test,  y_test,  augment=False)
print("\nCNN (piano-roll) windows")
print(f"  train: {Xcnn_train.shape}  val: {Xcnn_val.shape}  test: {Xcnn_test.shape}")
print(f"  files skipped (train/val/test): {sk4}/{sk5}/{sk6}")

### 6.2 Window-level class distribution and class weights

Windowing can change class balance because some composers have longer pieces (more windows). The counts below reflect the **actual training signal**. Class weights are computed from the training windows and passed to both models so that minority composers are not ignored.

In [ ]:
def show_distribution(y, title):
    counts = {IDX2LABEL[i]: int((y == i).sum()) for i in range(len(TARGET_COMPOSERS))}
    print(f"{title}: {counts}")
    return counts

print("Window counts per class")
_ = show_distribution(yseq_train, "LSTM train")
_ = show_distribution(ycnn_train, "CNN  train")

# Class weights (shared logic; computed separately per representation)
def class_weights(y):
    classes = np.arange(len(TARGET_COMPOSERS))
    w = compute_class_weight("balanced", classes=classes, y=y)
    return {int(c): float(wi) for c, wi in zip(classes, w)}

cw_seq = class_weights(yseq_train)
cw_cnn = class_weights(ycnn_train)
print("\nLSTM class weights:", {IDX2LABEL[k]: round(v, 3) for k, v in cw_seq.items()})
print("CNN  class weights:", {IDX2LABEL[k]: round(v, 3) for k, v in cw_cnn.items()})

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (y, t) in zip(ax, [(yseq_train, "LSTM train windows"),
                          (ycnn_train, "CNN train windows")]):
    vals = [ (y == i).sum() for i in range(len(TARGET_COMPOSERS)) ]
    a.bar(TARGET_COMPOSERS, vals, color=sns.color_palette("deep", 4))
    a.set_title(t); a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## 7. Model 1 - LSTM

The LSTM classifies a window of ordered pitch tokens. An `Embedding` layer maps each of the 129 possible tokens (128 pitches + 1 padding index) to a dense vector; two stacked LSTM layers model temporal dependencies; dropout regularizes; and a softmax `Dense` layer outputs a probability over the four composers.

**Key hyperparameters:** embedding dim 64, LSTM units 128 then 64, dropout 0.3, Adam optimizer, sparse categorical cross-entropy (labels are integers). These are explained in the report.

In [ ]:
def build_lstm(units1=128, units2=64, emb_dim=64, dropout=0.3, lr=1e-3):
    set_seeds()
    model = models.Sequential([
        layers.Input(shape=(SEQ_LEN,)),
        layers.Embedding(input_dim=PITCH_VOCAB + 1, output_dim=emb_dim,
                         mask_zero=False),
        layers.LSTM(units1, return_sequences=True, dropout=dropout),
        layers.LSTM(units2, dropout=dropout),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(len(TARGET_COMPOSERS), activation="softmax"),
    ], name="LSTM_composer_classifier")
    model.compile(optimizer=optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

build_lstm().summary()

## 8. Model 2 - CNN

The CNN classifies a piano-roll window (128 pitches x `PR_TIME` frames x 1 channel). Three convolutional blocks (Conv -> BatchNorm -> MaxPool) grow the receptive field; global average pooling collapses the feature map; and a softmax `Dense` layer outputs the composer probabilities. Batch normalization and dropout provide regularization.

**Key hyperparameters:** filters 32 -> 64 -> 128, 3x3 kernels, dropout 0.3, Adam optimizer, sparse categorical cross-entropy.

In [ ]:
def build_cnn(f1=32, f2=64, f3=128, dropout=0.3, lr=1e-3):
    set_seeds()
    model = models.Sequential([
        layers.Input(shape=(PR_PITCH, PR_TIME, 1)),
        layers.Conv2D(f1, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(f2, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(f3, 3, padding="same", activation="relu"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(len(TARGET_COMPOSERS), activation="softmax"),
    ], name="CNN_composer_classifier")
    model.compile(optimizer=optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

build_cnn().summary()

## 9. Training with callbacks

Both models use the same training routine: **early stopping** on validation loss (restores the best weights) and **model checkpointing** to save the best model. Class weights address imbalance. Training and validation loss/accuracy are recorded for later plotting.

In [ ]:
def train_model(build_fn, Xtr, ytr, Xval, yval, class_weight, tag, **kw):
    set_seeds()
    model = build_fn(**kw)
    ckpt_path = f"best_{tag}.keras"
    cbs = [
        callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                restore_best_weights=True),
        callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss",
                                  save_best_only=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                    patience=3, min_lr=1e-5),
    ]
    history = model.fit(
        Xtr, ytr, validation_data=(Xval, yval),
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        class_weight=class_weight, callbacks=cbs, verbose=2)
    return model, history

print("Training utility ready.")

## 10. Hyperparameter optimization

A small, honest grid search is run for each model over a few settings (learning rate and dropout). The configuration with the best **validation accuracy** is kept and later evaluated on the untouched test set. The grid is intentionally small so it finishes in a reasonable time; widen it if you have more compute.

> The printed table is produced at run time - copy its real values into the report's optimization table. No numbers are pre-filled.

In [ ]:
LSTM_GRID = [
    {"lr": 1e-3, "dropout": 0.3},
    {"lr": 5e-4, "dropout": 0.3},
    {"lr": 1e-3, "dropout": 0.5},
]
CNN_GRID = [
    {"lr": 1e-3, "dropout": 0.3},
    {"lr": 5e-4, "dropout": 0.3},
    {"lr": 1e-3, "dropout": 0.5},
]

def grid_search(build_fn, grid, Xtr, ytr, Xval, yval, cw, tag):
    rows, best = [], None
    for i, params in enumerate(grid):
        model, hist = train_model(build_fn, Xtr, ytr, Xval, yval, cw,
                                  tag=f"{tag}_cfg{i}", **params)
        val_acc = max(hist.history["val_accuracy"])
        rows.append({**params, "best_val_acc": round(val_acc, 4)})
        if best is None or val_acc > best["val_acc"]:
            best = {"val_acc": val_acc, "params": params,
                    "model": model, "history": hist}
    table = pd.DataFrame(rows)
    print(f"\n{tag} grid-search results:\n", table)
    print(f"Best {tag} config: {best['params']} "
          f"(val_acc={best['val_acc']:.4f})")
    return best, table

In [ ]:
# ---- Run LSTM search ----
best_lstm, lstm_grid_table = grid_search(
    build_lstm, LSTM_GRID, Xseq_train, yseq_train,
    Xseq_val, yseq_val, cw_seq, tag="lstm")
lstm_model, lstm_history = best_lstm["model"], best_lstm["history"]

In [ ]:
# ---- Run CNN search ----
best_cnn, cnn_grid_table = grid_search(
    build_cnn, CNN_GRID, Xcnn_train, ycnn_train,
    Xcnn_val, ycnn_val, cw_cnn, tag="cnn")
cnn_model, cnn_history = best_cnn["model"], best_cnn["history"]

## 11. Training and validation curves

The curves below show loss and accuracy for training vs. validation across epochs for each model. A growing gap between the two curves indicates overfitting; flat, low curves indicate underfitting. Interpret these in the report using the figures produced here.

In [ ]:
def plot_history(history, title):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="val")
    ax[0].set_title(f"{title} - loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="val")
    ax[1].set_title(f"{title} - accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_history(lstm_history, "LSTM")
plot_history(cnn_history, "CNN")

## 12. Evaluation

Each model is evaluated on the held-out **test** set. The evaluation reports overall accuracy, a full classification report (per-class precision, recall, F1), and a confusion matrix. Macro and weighted averages are both reported: **macro** treats every composer equally (fair under imbalance), while **weighted** reflects the test-set class frequencies.

In [ ]:
def evaluate(model, X, y, title):
    proba = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    y_pred = proba.argmax(axis=1)
    acc = accuracy_score(y, y_pred)
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y, y_pred, average="macro", zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y, y_pred, average="weighted", zero_division=0)
    print(f"===== {title} =====")
    print(f"Test accuracy      : {acc:.4f}")
    print(f"Macro    P/R/F1    : {p_macro:.4f} / {r_macro:.4f} / {f_macro:.4f}")
    print(f"Weighted P/R/F1    : {p_w:.4f} / {r_w:.4f} / {f_w:.4f}\n")
    print("Per-class classification report:")
    print(classification_report(y, y_pred, target_names=TARGET_COMPOSERS,
                                zero_division=0))
    cm = confusion_matrix(y, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=TARGET_COMPOSERS, yticklabels=TARGET_COMPOSERS)
    plt.title(f"{title} - confusion matrix")
    plt.xlabel("predicted"); plt.ylabel("true"); plt.tight_layout(); plt.show()
    return {"model": title, "accuracy": acc,
            "precision_macro": p_macro, "recall_macro": r_macro, "f1_macro": f_macro,
            "precision_weighted": p_w, "recall_weighted": r_w, "f1_weighted": f_w}

lstm_metrics = evaluate(lstm_model, Xseq_test, yseq_test, "LSTM")
cnn_metrics  = evaluate(cnn_model,  Xcnn_test,  ycnn_test,  "CNN")

## 13. Model comparison

The final comparison table is assembled directly from the metric dictionaries computed above, so it always reflects the real run. Use it as the source for the report's results table.

In [ ]:
comparison = pd.DataFrame([lstm_metrics, cnn_metrics]).set_index("model").round(4)
print("Model comparison (test set):\n")
print(comparison)

comparison[["accuracy", "f1_macro", "f1_weighted"]].plot(
    kind="bar", figsize=(8, 4))
plt.title("LSTM vs CNN - test metrics"); plt.ylabel("score")
plt.ylim(0, 1); plt.xticks(rotation=0); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

# Save the comparison so it can be pasted into the report.
comparison.to_csv("model_comparison.csv")
print("\nSaved model_comparison.csv")

## 14. Conclusions and future improvements

Summarize your findings **from the printed results above** (do not invent numbers). Points to address, using the figures and tables this notebook generated:

- **Which model performed better** overall (compare the test accuracy and macro-F1 in the comparison table).
- **Which composers were easiest to classify** and **which were most often confused** (read this from the two confusion matrices).
- **Overfitting vs. underfitting** signals (read this from the training/validation curves in Section 11).
- **Effect of class imbalance** (compare macro vs. weighted F1, and relate to the class-distribution charts).

**Possible future improvements**

- Richer LSTM tokens that combine pitch with duration and velocity, or an event-based encoding (note-on/note-off/time-shift).
- Multi-channel piano rolls (separate instruments) or log-scaled time resolution for the CNN.
- Stronger regularization or a hybrid CRNN (convolutions feeding an LSTM).
- More composers, cross-dataset validation, and k-fold cross-validation for tighter confidence intervals.
- Handling polyphony and voice separation more explicitly.

> **Reminder:** replace the bracketed placeholders in the report with the actual numbers this notebook prints. Do not state that a model is "highly accurate" unless the printed metrics support it.


## 15. References (APA 7)

Abadi, M., Agarwal, A., Barham, P., Brevdo, E., Chen, Z., Citro, C., ... Zheng, X. (2015). *TensorFlow: Large-scale machine learning on heterogeneous systems*. https://www.tensorflow.org/

Chollet, F. (2015). *Keras* [Computer software]. https://keras.io/

Fedorak, B. (2019). *midi_classic_music* [Data set]. Kaggle. https://www.kaggle.com/datasets/blanderbuss/midi-classic-music

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., ... Oliphant, T. E. (2020). Array programming with NumPy. *Nature, 585*(7825), 357-362. https://doi.org/10.1038/s41586-020-2649-2

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735-1780. https://doi.org/10.1162/neco.1997.9.8.1735

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90-95. https://doi.org/10.1109/MCSE.2007.55

LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE, 86*(11), 2278-2324. https://doi.org/10.1109/5.726791

McKinney, W. (2010). Data structures for statistical computing in Python. In S. van der Walt & J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56-61). https://doi.org/10.25080/Majora-92bf1922-00a

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825-2830.

Raffel, C., & Ellis, D. P. W. (2014). Intuitive analysis, creation and manipulation of MIDI data with pretty_midi. In *15th International Society for Music Information Retrieval Conference, Late-Breaking and Demo Session*.

Waskom, M. L. (2021). seaborn: Statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021
